#### In this notebook, I extended the FUSE algorithm [[paper](https://arxiv.org/pdf/2510.11250)] for node embedding generation from structural data alone to the OGBN-Products dataset [[dataset](https://ogb.stanford.edu/docs/nodeprop/#ogbn-products)].

## Results Summary

The generated FUSE embeddings were evaluated using two downstream classifiers:

1. **MLP**
2. **Multinomial Logistic Regression**

The evaluation was performed under two different split settings:

1. **30-70 MCAR split**  
   - 30% nodes used for training  
   - 70% nodes used for testing  
   - Labels are masked completely at random, following the FUSE paper-style split.

2. **Official OGBN-Products split**  
   - Train/validation/test split provided by OGBN.

---

### Classification Performance

| Split Type | Classifier | Test Accuracy |  Avg F1 Score |
|---|---:|---:|---:|
| 30-70 MCAR | MLP | **0.8008** | **0.7826** |
| 30-70 MCAR | Multinomial Logistic Regression | **0.7686** | **0.7474** |
| OGBN Official Split | MLP | **0.6743** | **0.6731** |
| OGBN Official Split | Multinomial Logistic Regression | **0.6736** | **0.6758** |

---

### Key Observation

The FUSE embeddings perform significantly better under the **30-70 MCAR split** compared to the official OGBN split. This is expected because the MCAR split randomly samples train and test nodes from the same overall node distribution, while the official OGBN split is more challenging and distribution-shifted.




In [1]:
!pip install ogb
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.1 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23

### Importing the libraries :

In [2]:

import torch
import numpy as np
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_torch_csr_tensor   

from ogb.nodeproppred import NodePropPredDataset 


import wandb 
from sklearn.linear_model import LogisticRegression

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


### Importing OGBN Products Dataset :

In [3]:

original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return original_torch_load(*args, **kwargs)

torch.load = patched_torch_load

In [4]:
dataset = NodePropPredDataset(name='ogbn-products')

graph, labels = dataset[0]

graph

This will download 1.38GB. Will you proceed? (y/N)
 y


Downloaded 1.38 GB: 100%|██████████| 1414/1414 [01:08<00:00, 20.54it/s]


Extracting dataset/products.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


Saving...


{'edge_index': array([[      0,  152857,       0, ..., 2449028,   53324, 2449028],
        [ 152857,       0,   32104, ...,  162836, 2449028,   53324]],
       shape=(2, 123718280)),
 'edge_feat': None,
 'node_feat': array([[ 0.03193326, -0.1958605 ,  0.0519961 , ...,  0.07669606,
         -0.3929545 , -0.06478424],
        [-0.02405794,  0.63032097,  1.0605699 , ..., -1.6874819 ,
          3.5866776 ,  0.818219  ],
        [ 0.33269015, -0.5585958 , -0.28860757, ..., -0.37157044,
          0.2520575 ,  0.04153213],
        ...,
        [ 0.10660695,  0.2654852 , -0.00567423, ...,  1.0867023 ,
          0.07590195, -1.1736895 ],
        [ 0.24968362, -0.25740346,  0.41230008, ...,  1.5465808 ,
          1.0309792 , -0.29657176],
        [ 0.7175324 , -0.23930131,  0.04430327, ..., -1.0132493 ,
         -0.41407427, -0.08227058]], shape=(2449029, 100), dtype=float32),
 'num_nodes': 2449029}

In [15]:

edge_index = torch.tensor(graph['edge_index'], dtype=torch.long)
node_features = torch.tensor(graph['node_feat'], dtype=torch.float)
labels_tensor = torch.tensor(labels, dtype=torch.long).squeeze()



num_nodes = graph['num_nodes']
num_classes = dataset.num_classes

print(f"Nodes: {num_nodes}, Edges: {edge_index.shape[1]}, Classes: {num_classes}")

print(F"Node_features_dim:{node_features.shape}")

Nodes: 2449029, Edges: 123718280, Classes: 47
Node_features_dim:torch.Size([2449029, 100])


In [16]:
# torch.save(edge_index, "Products_ogbn_edge_index")
# torch.save(node_features, "Products_ogbn_node_features")
# torch.save(labels_tensor, "Products_ogbn_labels_tensor")



In [17]:
split_idx = dataset.get_idx_split()

train_idx = split_idx['train']
valid_idx = split_idx['valid']
test_idx = split_idx['test']

# torch.save(train_idx, "Products_ogbn_train_index")
# torch.save(valid_idx, "Products_ogbn_valid_index")
# torch.save(test_idx, "Products_ogbn_test_index")


In [18]:
# import json
# import os

# dataset_metadata = {
#     "title": "ogbn-products-metadataokayy",
#     "id": "mehulgoyal1729/ogbn-products-metadataokayy",
#     "licenses": [{"name": "CC0-1.0"}]
# }

# with open("/kaggle/working/dataset-metadata.json", "w") as f:
#     json.dump(dataset_metadata, f)

In [19]:
# !kaggle datasets files mehulgoyal1729/ogbn-products-metadataokayy

In [20]:
n = num_nodes 

is_labeled = torch.zeros(n, dtype=torch.bool, device=device)

is_labeled[train_idx] = True

is_masked = ~is_labeled 

labels_masked = labels_tensor.clone().to(device)
labels_masked[is_masked] = -1

print(f"Total Nodes: {n}")
print(f"Labeled (Train): {is_labeled.sum().item()}")
print(f"Masked (Valid + Test): {is_masked.sum().item()}")

Total Nodes: 2449029
Labeled (Train): 196615
Masked (Valid + Test): 2252414


In [21]:
print(f"Train_idx_count:{len(train_idx)}")
print(f"valid_idx_count:{len(valid_idx)}")
print(f"test_idx_count:{len(test_idx)}")

Train_idx_count:196615
valid_idx_count:39323
test_idx_count:2213091


In [22]:

labels_masked = torch.full((n,), -1, dtype=torch.long, device=device)
labels_masked[train_idx] = labels_tensor[train_idx].to(device)
is_masked = (labels_masked == -1)

print(f"Masked Labels: {labels_masked.shape}")
print(f"Labeled nodes (Train): {(labels_masked != -1).sum().item()}")
print(f"Unlabeled nodes (Valid + Test): {is_masked.sum().item()}")

Masked Labels: torch.Size([2449029])
Labeled nodes (Train): 196615
Unlabeled nodes (Valid + Test): 2252414


In [23]:
print(f"Original Feature Matrix Shape: {node_features.shape}")

Original Feature Matrix Shape: torch.Size([2449029, 100])


## Building sparse adjacency matrix:


In [46]:
n = num_nodes
m = edge_index.shape[1] // 2  # to prevent double counting of edgees

edge_index

tensor([[      0,  152857,       0,  ..., 2449028,   53324, 2449028],
        [ 152857,       0,   32104,  ...,  162836, 2449028,   53324]])

In [47]:
# self edges removal  :
mask = edge_index[0] != edge_index[1]
edge_index_clean = edge_index[:, mask]


#     COO to  CSR
vals = torch.ones(edge_index.shape[1], dtype=torch.float32)
A_sparse = torch.sparse_coo_tensor(edge_index, vals, (n, n)).to_sparse_csr().to(device)
deg = torch.zeros(n, dtype=torch.float32, device=device)
deg.scatter_add_(0, edge_index[0].to(device), torch.ones(edge_index.shape[1], device=device))
d = deg.unsqueeze(1) 

labels = labels_tensor
print(f'A_sparse: {A_sparse.shape}, nnz={A_sparse._nnz()}, d: {d.shape}')

KeyboardInterrupt: 

### Initialising embeddings:

#### Hyperparams chosen on the basis of the optimal hyperparameters in the original FUSE paper for Arxiv:

- Embedding dimension = 100
- lr = 0.05
- Iterations  = 200



In [ ]:

S = torch.randn(n, 100, device=device)

S_cpu = S.cpu()
del S 
torch.cuda.empty_cache()

Q_cpu, _ = torch.linalg.qr(S_cpu, mode='reduced')

S = Q_cpu.to(device)
del Q_cpu


In [ ]:
def build_csr_graph(edge_index: torch.Tensor, n: int, device):
    """
     edge list to 1D CSR format arrays.
    """
    src, dst = edge_index[0], edge_index[1]
    
    order = torch.argsort(src)
    src_s, dst_s = src[order], dst[order]
    
    deg = torch.bincount(src_s, minlength=n).to(device)
    
    row_ptr = torch.zeros(n + 1, dtype=torch.long, device=device)
    row_ptr[1:] = torch.cumsum(deg, dim=0)
    
    col_idx = dst_s.to(device)
    
    return row_ptr, col_idx, deg

row_ptr, col_idx, deg_vec = build_csr_graph(edge_index_clean, n, device)
print(f"Row Pointers: {row_ptr.shape}, Flattened Edges: {col_idx.shape}")

Row Pointers: torch.Size([2449030]), Flattened Edges: torch.Size([123718024])


## Batched CSR adajcency matrix based Random Walk  :

In [ ]:


def batched_csr_random_walk(row_ptr, col_idx, deg_vec, labels_masked, is_masked, 
    r=20, L=4, Lp=1, # based on original FUSE paper 
    device='cpu'
):

    unlab_ids = torch.where(is_masked)[0]     
    n_unlab   = unlab_ids.shape[0]

    walk_neigh = torch.full((n_unlab, r, Lp), -1, dtype=torch.long, device=device)
    labeled_count = torch.zeros(n_unlab, r, dtype=torch.long, device=device)

    current = unlab_ids.unsqueeze(1).expand(n_unlab, r).clone() 

    for step in range(L):
        deg_curr = deg_vec[current].clamp(min=1) 

        offsets = (torch.rand_like(current, dtype=torch.float) * deg_curr).long()

        flat_indices = row_ptr[current] + offsets

        next_node = col_idx[flat_indices]

        next_is_lab = labels_masked[next_node] != -1                  
        slot_open   = labeled_count < Lp                             
        record      = next_is_lab & slot_open                       

        lc_clamped = labeled_count.clamp(max=Lp - 1) 
        
        walk_neigh.scatter_(
            2,
            lc_clamped.unsqueeze(2),
            torch.where(record.unsqueeze(2), next_node.unsqueeze(2), walk_neigh.gather(2, lc_clamped.unsqueeze(2))))
        
        labeled_count = labeled_count + record.long()
        current = next_node

    return walk_neigh, unlab_ids

walk_neigh, unlab_ids = batched_csr_random_walk(
    row_ptr, col_idx, deg_vec, labels_masked, is_masked,
    r=20, L=4, Lp=1, device=device
)

print('walk_neigh:', walk_neigh.shape, '| unlab_ids:', unlab_ids.shape)

walk_neigh: torch.Size([2252414, 20, 1]) | unlab_ids: torch.Size([2252414])


## FUSE Optimization:

### Modularity Optimzation :

In [ ]:
def modularity_loss(A_sparse, d, S, m, grad_total):
    if d.dim() == 1:
        d = d.unsqueeze(1).float()
    elif d.dtype != S.dtype:
        d = d.float()
        
    # Sparse Mat-Mul:
    grad_total.copy_(torch.sparse.mm(A_sparse, S))
    
    # degree correction:
    dS = d.t() @ S
    scaled_d = d * (1.0 / (2 * m))
    
    # Subtracting from the total grad loss :
    grad_total.addmm_(scaled_d, dS, alpha=-1.0, beta=1.0)


In [ ]:


def supervised_loss(S, labels_masked, is_labeled, num_classes, grad_total, lambda_sup):
    lab_idx  = torch.where(is_labeled)[0]
    lab_cls  = labels_masked[lab_idx]
    lab_emb  = S[lab_idx]

    class_sum   = torch.zeros(num_classes, S.shape[1], device=S.device)
    class_count = torch.zeros(num_classes, device=S.device)
    
    class_sum.scatter_add_(0, lab_cls.unsqueeze(1).expand_as(lab_emb), lab_emb)
    class_count.scatter_add_(0, lab_cls, torch.ones(lab_idx.shape[0], device=S.device))

    mu = class_sum / class_count.unsqueeze(1).clamp(min=1) # class means 
    
    delta = lab_emb - mu[lab_cls]
    
    grad_total[lab_idx] -= lambda_sup * delta

In [ ]:



def semi_sup_loss(S, walk_neigh, unlab_ids, grad_total, lambda_semi, batch_size=25000):
    n_unlab = unlab_ids.shape[0]
    P = walk_neigh.shape[1] * walk_neigh.shape[2]

    flat_neigh = walk_neigh.view(n_unlab, P)    
    valid_mask = (flat_neigh >= 0)              
    safe_neigh = flat_neigh.clamp(min=0)        

    for i in range(0, n_unlab, batch_size):
        end = min(i + batch_size, n_unlab)
        ids_batch = unlab_ids[i:end]
        safe_neigh_batch = safe_neigh[i:end]
        valid_mask_batch = valid_mask[i:end]

        S_i = S[ids_batch]               
        S_j = S[safe_neigh_batch]        

        sim = torch.bmm(S_i.unsqueeze(1), S_j.transpose(1, 2)).squeeze(1)  # structural similarity with candidate leablled nodes
        sim = sim.masked_fill(~valid_mask_batch, float('-inf'))

        w = torch.softmax(sim, dim=1)               
        w = torch.nan_to_num(w, nan=0.0)            

        weighted_sum = torch.bmm(w.unsqueeze(1), S_j).squeeze(1)   

        has_valid = valid_mask_batch.any(dim=1)           
        delta = torch.where(has_valid.unsqueeze(1), S_i - weighted_sum, torch.zeros_like(S_i))

        grad_total[ids_batch] -= lambda_semi * delta
        
        del S_i, S_j, sim, w, weighted_sum, delta

###  Optimization: 

In [ ]:
import time
import torch
import wandb

# FUSE Original Relative Hyperparameters:
lr          = 0.05
lambda_mod  = 1.0
lambda_sup  = 1.0
lambda_semi = 1.9
T_epochs    = 200

wandb.init(
    project="fuse-ogbn-products",
    name="phase1-unit-norm-equalization",
    config={"lr": lr, "lambda_mod": lambda_mod, "lambda_sup": lambda_sup, "lambda_semi": lambda_semi}
)

print("\n--- Starting Phase 1: Unit-Norm Equalized Structural Generation ---")
t0 = time.time()

with torch.no_grad():
    grad_total = torch.zeros_like(S)
    
    grad_temp = torch.zeros_like(S) 
    
    for iteration in range(T_epochs):
        grad_total.zero_()
        
        
        
        grad_scratch = torch.zeros_like(S) 
        
        

        # ==========================================
        # 1. Modularity
        # ==========================================
        grad_temp.zero_()
        modularity_loss(A_sparse, d, S, m, grad_scratch)
        
        norm_mod = torch.norm(grad_temp).item()
        if norm_mod > 0:
            grad_scratch.div_(norm_mod) # saving the normalised value 
        grad_total.add_(grad_temp, alpha=lambda_mod) 
            
        # ==========================================
        # 2. Supervised
        # ==========================================
        grad_scratch.zero_()
        # Pass lambda_sup=1.0 to the function, we scale it externally
        supervised_loss(S, labels_masked, is_labeled, num_classes, grad_scratch, lambda_sup=1.0)
        
        norm_sup = torch.norm(grad_scratch).item()
        if norm_sup > 0:
            grad_scratch.div_(norm_sup) # Squash to length 1.0
        grad_total.add_(grad_scratch, alpha=lambda_sup)
            
        # ==========================================
        # 3. Semi-Supervised
        # ==========================================
        grad_scratch.zero_()
        # Pass lambda_semi=1.0 to the function, we scale it externally
        semi_sup_loss(S, walk_neigh, unlab_ids, grad_scratch, lambda_semi=1.0)
        
        norm_semi = torch.norm(grad_scratch).item()
        if norm_semi > 0:
            grad_scratch.div_(norm_semi) # Squash to length 1.0
        grad_total.add_(grad_scratch, alpha=lambda_semi)
            
        S.add_(grad_total, alpha=lr)
        
        # Orthogonalization:
        Q, _ = torch.linalg.qr(S, mode='reduced')
        S.copy_(Q) 
        del Q
        torch.cuda.empty_cache()

        total_grad_norm = torch.norm(grad_total).item()
        elapsed = time.time() - t0
        
        wandb.log({
            "epoch": iteration + 1,
            "Mod_RawNorm": norm_mod,
            "Sup_RawNorm": norm_sup,
            "Semi_RawNorm": norm_semi,
            "Step_Vector_Norm": total_grad_norm
        })

        if (iteration + 1) % 1 == 0 or iteration == 0:
            print(f"Iter {iteration+1:3d}/{T_epochs} | "
                  f"Mod: {norm_mod:8.2f} | "
                  f"Sup: {norm_sup:8.2f} | "
                  f"Semi: {norm_semi:8.2f} | "
                  f"Step Norm: {total_grad_norm:4.2f} | "
                  f"Time: {elapsed:.1f}s")

wandb.finish()

print(f'\nTotal structural generation time: {time.time()-t0:.2f}s')
print("Phase 1 Complete. Structural embeddings optimized and saved.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mehul22 (mehul22-iiser-thiruvananthapuram) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



--- Starting Phase 1: Unit-Norm Equalized Structural Generation ---
Iter   1/200 | Mod:    71.09 | Sup:     2.83 | Semi:    10.30 | Step Norm: 2.41 | Time: 9.2s
Iter   2/200 | Mod:    72.69 | Sup:     2.81 | Semi:    10.26 | Step Norm: 2.40 | Time: 18.1s
Iter   3/200 | Mod:    74.63 | Sup:     2.79 | Semi:    10.23 | Step Norm: 2.39 | Time: 27.0s
Iter   4/200 | Mod:    76.93 | Sup:     2.76 | Semi:    10.20 | Step Norm: 2.38 | Time: 35.8s
Iter   5/200 | Mod:    79.65 | Sup:     2.74 | Semi:    10.17 | Step Norm: 2.37 | Time: 44.7s
Iter   6/200 | Mod:    82.83 | Sup:     2.72 | Semi:    10.14 | Step Norm: 2.37 | Time: 53.7s
Iter   7/200 | Mod:    86.53 | Sup:     2.69 | Semi:    10.12 | Step Norm: 2.36 | Time: 62.6s
Iter   8/200 | Mod:    90.81 | Sup:     2.67 | Semi:    10.09 | Step Norm: 2.35 | Time: 71.6s
Iter   9/200 | Mod:    95.70 | Sup:     2.65 | Semi:    10.07 | Step Norm: 2.35 | Time: 80.6s
Iter  10/200 | Mod:   101.27 | Sup:     2.63 | Semi:    10.05 | Step Norm: 2.34 | Time

Mod_RawNorm,▁▂▂▂▂▃▃▃▄▄▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████████████
Semi_RawNorm,█████████▇▇▇▇▇▇▅▅▅▅▅▅▅▅▅▅▅▅▅▄▄▄▃▃▂▂▂▂▁▁▁
Step_Vector_Norm,██████████▇▇▇▇▆▅▅▅▅▅▄▃▃▂▂▂▇▂▂▂▇▇▁▁▁▇▇▁▁▁
Sup_RawNorm,██▇▇▇▇▆▆▆▆▅▅▄▄▄▃▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇█
Mod_RawNorm,920.92218
Semi_RawNorm,4.73585
Step_Vector_Norm,1.73431
Sup_RawNorm,0.06045
epoch,200



Total structural generation time: 1837.16s
Phase 1 Complete. Structural embeddings optimized and saved.


In [26]:

torch.save(S, "S_products_normed_k100")

In [33]:
del A_sparse

In [34]:
S_given = node_features

In [35]:
S_given.shape

torch.Size([2449029, 100])

## For testing over FUSE Embeddings :

### MLP: (based on OGBN Benchmark paper)

In [24]:
S = torch.load("/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100")

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S_generated = S
train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = labels_tensor.to(device) 

S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()

print(f"Train nodes: {S_train.shape[0]} | Valid nodes: {S_valid.shape[0]} | Test nodes: {S_test.shape[0]}")

mean = S_train.mean(dim=0, keepdim=True)
std  = S_train.std(dim=0, keepdim=True) + 1e-7

S_train_scaled = (S_train - mean) / std
S_valid_scaled = (S_valid - mean) / std
S_test_scaled  = (S_test - mean) / std

batch_size = 4096
train_dataset = TensorDataset(S_train_scaled, y_train)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
        nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5), 
            
            nn.Linear(hidden, hidden), 
        nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x): return self.net(x)


K = S_train.shape[1]
model = MLP(K, 256, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

epochs = 200
losses = []
best_val_acc = 0.0
best_model_weights = None
patience = 20  #
epochs_no_improve = 0
target_threshold = 0.95

for epoch in range(epochs):
    
    model.train()
    epoch_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    avg_train_loss = epoch_loss / len(train_loader)
    losses.append(epoch_loss)
    
    model.eval()
    with torch.no_grad():
        
        val_logits = model(S_valid_scaled)
        val_preds = val_logits.argmax(dim=1)
        val_correct = (val_preds == y_valid).sum().item()
        val_acc = val_correct / len(y_valid)

    print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if val_acc >= target_threshold:
        print(f"\nTarget Validation Accuracy ({target_threshold}) reached! Stopping early.")
        break
        
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
        break

from sklearn.metrics import classification_report, accuracy_score, f1_score

print(f"\nLoading best model weights (Val Acc: {best_val_acc:.4f}) for Final Testing...")
model.load_state_dict(best_model_weights)
model.eval()

with torch.no_grad():
    test_logits = model(S_test_scaled)
    test_preds = test_logits.argmax(dim=1)
    test_correct = (test_preds == y_test).sum().item()
    test_acc = test_correct / len(y_test)
    
print(f"Final OGBN-Products Test Accuracy: {test_acc:.4f}")
print("\n--- MLP Classification Report ---")
print(classification_report(
    y_test.detach().cpu().numpy(),
    test_preds.detach().cpu().numpy(),
    digits=4
))



Loading best model weights (Val Acc: 0.8354) for Final Testing...
Final OGBN-Products Test Accuracy: 0.6743

--- MLP Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.5501    0.7440    0.6325    100670
           1     0.6352    0.7524    0.6889     95236
           2     0.8158    0.8033    0.8095    102870
           3     0.6576    0.6655    0.6615    135255
           4     0.8381    0.7863    0.8113    596195
           5     0.6651    0.8517    0.7469     35985
           6     0.7492    0.7078    0.7279    140356
           7     0.9115    0.8880    0.8996    152202
           8     0.6583    0.9452    0.7761     97679
           9     0.8492    0.8180    0.8333     59852
          10     0.6370    0.7840    0.7029     47165
          11     0.0818    0.3000    0.1285     28933
          12     0.7370    0.4623    0.5682    128632
          13     0.7390    0.7442    0.7416     88694
          14     0.0784    0.4181    0.1320      2734
          15     0.6780    0.9080    0.7763     23871
          16     0.7632    0.1781    0.2888     83019
          17     0.8551    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Logistic Regression



In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("\n--- Transferring Data to CPU for Scikit-Learn ---")
X_train_np = S_train_scaled.cpu().numpy()
y_train_np = y_train.cpu().numpy()

X_test_np  = S_test_scaled.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

print("Training Logistic Regression...")
clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
clf.fit(X_train_np, y_train_np)

print("Evaluating on Test Set...")
y_pred = clf.predict(X_test_np)


print(f"Accuracy: {accuracy_score(y_pred,y_test_np )}")


print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test_np, y_pred, digits=4))


--- Transferring Data to CPU for Scikit-Learn ---
Training Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating on Test Set...
Accuracy: 0.6735502516615901

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.5939    0.7317    0.6557    100670
           1     0.6757    0.7376    0.7053     95236
           2     0.7528    0.8186    0.7843    102870
           3     0.6605    0.6329    0.6464    135255
           4     0.8639    0.8055    0.8337    596195
           5     0.6971    0.8117    0.7501     35985
           6     0.7204    0.7049    0.7126    140356
           7     0.8818    0.9052    0.8933    152202
           8     0.6960    0.9242    0.7940     97679
           9     0.8730    0.8222    0.8468     59852
          10     0.6601    0.6689    0.6645     47165
          11     0.0545    0.3971    0.0958     28933
          12     0.7352    0.4943    0.5911    128632
          13     0.7010    0.7339    0.7171     88694
          14     0.1288    0.4192    0.1971      2734
          15     0.5684    0.9258    0.7043     23871
          16     0.7569    0.1403    0.2368     83019
          17     0.8606    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##  30-70 Dataset Split


In [29]:
S = torch.load("/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100")

In [31]:

seed = 42
train_ratio = 0.30

S_eval = S.detach()
labels_eval = labels_tensor.to(device).long()

n = S_eval.shape[0]
num_train = int(train_ratio * n)

generator_cpu = torch.Generator(device='cpu')
generator_cpu.manual_seed(seed)

perm_cpu = torch.randperm(n, generator=generator_cpu)
train_idx_cpu = perm_cpu[:num_train]
test_idx_cpu  = perm_cpu[num_train:]

train_idx = train_idx_cpu.to(device)
test_idx  = test_idx_cpu.to(device)

print("--- 30-70 Dataset Split ---")
print(f"Total nodes: {n}")
print(f"Train nodes: {len(train_idx_cpu)} ({len(train_idx_cpu)/n:.2%})")
print(f"Test nodes:  {len(test_idx_cpu)} ({len(test_idx_cpu)/n:.2%})")

labels_cpu_for_counts = labels_eval.detach().cpu()
train_class_counts = torch.bincount(labels_cpu_for_counts[train_idx_cpu], minlength=num_classes)
test_class_counts  = torch.bincount(labels_cpu_for_counts[test_idx_cpu], minlength=num_classes)

print(f"Classes present in train: {(train_class_counts > 0).sum().item()} / {num_classes}")
print(f"Classes present in test:  {(test_class_counts > 0).sum().item()} / {num_classes}")

X_train = S_eval[train_idx].detach()
y_train = labels_eval[train_idx].detach()

X_test = S_eval[test_idx].detach()
y_test = labels_eval[test_idx].detach()

mean = X_train.mean(dim=0, keepdim=True)
std  = X_train.std(dim=0, keepdim=True) + 1e-7

X_train_scaled = (X_train - mean) / std
X_test_scaled  = (X_test  - mean) / std

print(f"Train tensor: {X_train_scaled.shape}")
print(f"Test tensor:  {X_test_scaled.shape}")


--- 30-70 Dataset Split ---
Total nodes: 2449029
Train nodes: 734708 (30.00%)
Test nodes:  1714321 (70.00%)
Classes present in train: 46 / 47
Classes present in test:  47 / 47
Train tensor: torch.Size([734708, 100])
Test tensor:  torch.Size([1714321, 100])


### MLP: (based on OGBN Benchmark paper)

In [33]:

import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

valid_ratio = 0.10
train_count = X_train_scaled.shape[0]
internal_val_count = max(num_classes, int(valid_ratio * train_count))

inner_perm_cpu = torch.randperm(train_count, generator=generator_cpu)
inner_val_pos = inner_perm_cpu[:internal_val_count].to(device)
inner_train_pos = inner_perm_cpu[internal_val_count:].to(device)

X_inner_train = X_train_scaled[inner_train_pos]
y_inner_train = y_train[inner_train_pos].long()

X_inner_val = X_train_scaled[inner_val_pos]
y_inner_val = y_train[inner_val_pos].long()


batch_size = 4096
train_dataset = TensorDataset(X_inner_train, y_inner_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

K = X_train_scaled.shape[1]
model = MLP(K, 256, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

epochs = 200
patience = 10
best_val_acc = 0.0
best_model_weights = None
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_inner_val)
        val_preds = val_logits.argmax(dim=1)
        val_acc = (val_preds == y_inner_val).float().mean().item()

    print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"\nMLP early stopping triggered after {patience} epochs without improvement.")
        break

print(f"\nLoading best MLP weights with inner validation accuracy: {best_val_acc:.4f}")
model.load_state_dict(best_model_weights)
model.eval()

with torch.no_grad():
    test_logits = model(X_test_scaled)
    mlp_test_preds = test_logits.argmax(dim=1)

mlp_test_acc = (mlp_test_preds == y_test).float().mean().item()
mlp_test_f1 = f1_score(
    y_test.detach().cpu().numpy(),
    mlp_test_preds.detach().cpu().numpy(),
    average='weighted'
)

print("\n--- 30-70 MLP Test Results ---")
print(f"MLP Test Accuracy:    {mlp_test_acc:.4f}")
print(f"MLP Weighted Test F1: {mlp_test_f1:.4f}")

print("\n--- MLP Classification Report ---")
print(classification_report(
    y_test.detach().cpu().numpy(),
    mlp_test_preds.detach().cpu().numpy(),
    digits=4
))

Epoch   1/200 | Train Loss: 1.0107 | Val Acc: 0.8007
Epoch   2/200 | Train Loss: 0.8999 | Val Acc: 0.8033
Epoch   3/200 | Train Loss: 0.8959 | Val Acc: 0.8031
Epoch   4/200 | Train Loss: 0.8937 | Val Acc: 0.8032
Epoch   5/200 | Train Loss: 0.8925 | Val Acc: 0.8041
Epoch   6/200 | Train Loss: 0.8912 | Val Acc: 0.8041
Epoch   7/200 | Train Loss: 0.8902 | Val Acc: 0.8025
Epoch   8/200 | Train Loss: 0.8898 | Val Acc: 0.8037
Epoch   9/200 | Train Loss: 0.8894 | Val Acc: 0.8041
Epoch  10/200 | Train Loss: 0.8893 | Val Acc: 0.8026
Epoch  11/200 | Train Loss: 0.8896 | Val Acc: 0.8039
Epoch  12/200 | Train Loss: 0.8884 | Val Acc: 0.8035
Epoch  13/200 | Train Loss: 0.8891 | Val Acc: 0.8039
Epoch  14/200 | Train Loss: 0.8882 | Val Acc: 0.8031
Epoch  15/200 | Train Loss: 0.8876 | Val Acc: 0.8035

MLP early stopping triggered after 10 epochs without improvement.

Loading best MLP weights with inner validation accuracy: 0.8041

--- 30-70 MLP Test Results ---
MLP Test Accuracy:    0.8008
MLP Weighted

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.7193    0.7774    0.7472     79889
           1     0.7470    0.7995    0.7724     77006
           2     0.8692    0.8898    0.8794     81230
           3     0.7218    0.8042    0.7607    105684
           4     0.8536    0.9446    0.8968    468023
           5     0.7777    0.8568    0.8153     28440
           6     0.7744    0.8744    0.8214    111218
           7     0.9134    0.9325    0.9229    120466
           8     0.8095    0.9254    0.8636     77635
           9     0.8802    0.8856    0.8829     47301
          10     0.7002    0.8267    0.7582     36586
          11     0.5149    0.2526    0.3389     23067
          12     0.7563    0.7019    0.7281     92192
          13     0.8341    0.7815    0.8070     70792
          14     0.5789    0.0051    0.0101      2157
          15     0.8305    0.9288    0.8769     18872
          16     0.7086    0.4317    0.5365     58691
          17     0.8755    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Logistic Regression:

In [34]:

X_train_np = X_train_scaled.detach().cpu().numpy()
y_train_np = y_train.detach().cpu().numpy()

X_test_np = X_test_scaled.detach().cpu().numpy()
y_test_np = y_test.detach().cpu().numpy()

print("Training Logistic Regression on 30% MCAR train nodes...")
clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
clf.fit(X_train_np, y_train_np)

print("Evaluating Logistic Regression on 70% MCAR test nodes...")
logreg_pred = clf.predict(X_test_np)

logreg_acc = accuracy_score(y_test_np, logreg_pred)
logreg_f1 = f1_score(y_test_np, logreg_pred, average='weighted')

print("\n--- 30-70 MCAR Logistic Regression Test Results ---")
print(f"MCAR Logistic Regression Test Accuracy:    {logreg_acc:.4f}")
print(f"MCAR Logistic Regression Weighted Test F1: {logreg_f1:.4f}")

print("\n--- MCAR Logistic Regression Classification Report ---")
print(classification_report(y_test_np, logreg_pred, digits=4))

Training Logistic Regression on 30% MCAR train nodes...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating Logistic Regression on 70% MCAR test nodes...

--- 30-70 MCAR Logistic Regression Test Results ---
MCAR Logistic Regression Test Accuracy:    0.7686
MCAR Logistic Regression Weighted Test F1: 0.7474

--- MCAR Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.6606    0.7610    0.7073     79889
           1     0.7146    0.8081    0.7585     77006
           2     0.7975    0.8739    0.8340     81230
           3     0.7268    0.7732    0.7493    105684
           4     0.7934    0.8929    0.8402    468023
           5     0.7831    0.8448    0.8128     28440
           6     0.7550    0.8777    0.8117    111218
           7     0.9137    0.9257    0.9197    120466
           8     0.7672    0.9300    0.8408     77635
           9     0.8771    0.8729    0.8750     47301
          10     0.7391    0.7958    0.7664     36586
          11     0.0100    0.0004    0.0008     23067
          12     0.7379    0.6436    0.6875     92192
          13     0.7802    0.7217    0.7498     70792
          14     0.0000    0.0000    0.0000      2157
          15     0.8227    0.9040    0.8614     18872
          16     0.6928    0.4177    0.5212     58691
          17     0.8911    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
